# 01. Integral norms: estimators and convergence

**Scope.** Estimating an integral norm of a path from discrete samples, and understanding which measure on time a sampled loss approximates. This is the mathematical foundation for the completed comparisons in notebooks 05 and 06.

---

## What to run

```bash
pytest -q tests/test_norms.py
```

Notebook has no code cells. Every claim below is either proved in text or asserted in `tests/test_norms.py`, so running those tests is what checks it. Named tests are cited beside the claims they cover.


## 0. Setting

### 0.1 Problem

Let $f:[0,T]\to\mathbb{R}^d$ be sampled at $t_0 < \cdots < t_N$: $N+1$ points, $N$ intervals. Estimate

$$\|f\|_{L^p} = \Big(\int_0^T |f|^p\Big)^{1/p} .$$

Write $g = |f|^p$ for the **integrand**, so the target and its estimator are

$$I(g) = \int_0^T g , \qquad Q_w(g) = \sum_{i=0}^N w_i\, g(t_i) .$$

Error bounds constrain $g$, not $f$. For odd $p$, $g$ has a kink at every zero of $f$ however smooth $f$ is.

Nodes are given, so what remains is choice of $w$. Two choices recur:

$$w^{\mathrm{trap}}_i = \tfrac12(\Delta t_{i-1} + \Delta t_i), \qquad w^{\mathrm{unif}}_i = \tfrac{T}{N+1},
\qquad \Delta t_i = t_{i+1}-t_i \ \ (0 \le i \le N-1) ,$$

with convention $\Delta t_{-1} = \Delta t_N = 0$, so $w^{\mathrm{trap}}_0 = \tfrac12\Delta t_0$ and $w^{\mathrm{trap}}_N = \tfrac12\Delta t_{N-1}$. Each node carries length of interval for which it is nearest; endpoints have interval on one side only. Both vectors sum to $T$.

Pointwise MSE is the uniform-weight rule:

$$\mathrm{MSE} = \frac{1}{N+1}\sum_i f(t_i)^2 = \tfrac1T\,Q^{\mathrm{unif}}(f^2) .$$

§2 and §3 ask when each converges to $I(g)$. In loss application $f = \hat y - y$.

### 0.2 Relation to the baseline loss

On a **uniform** grid $w^{\mathrm{unif}}$ and $w^{\mathrm{trap}}$ differ only at the two endpoints (§1.2), so both converge and

$$\mathrm{MSE} \;\xrightarrow[N\to\infty]{}\; \frac1T\int_0^T f^2 \;=\; \frac1T\|f\|_{L^2}^2 .$$

Integral norm is continuum limit of MSE. §3 removes uniformity.

### 0.3 Contents

All of it is standard. Verification is in `tests/`.

| section | | result |
|---|---|---|
| §1 | Choice of weights | trapezoid |
| §2 | Convergence rates | quadrature error is not a confounder |
| §3 | Non-uniform sampling | MSE converges to the wrong limit |
| §4 | Choice of $p$ | discussion fixes $p=2$; code does not |
| §5 | Summary | |

ML literature on irregular sampling (Che et al., 2018; Rubanova et al., 2019; Shukla & Marlin, 2021; Kidger et al., 2020) modifies model to ingest irregular observations and leaves objective at pointwise MSE. §3 shows that this choice of objective changes the limit under non-uniform sampling.

> **Parallel with sFML.** There, a uniform average over a non-uniformly visited *state* space; here, over a non-uniformly sampled *time* axis.

## 1. Choice of weights

### 1.1 Quadrature rules

Each rule replaces $g$ on a subinterval by an interpolating polynomial and integrates that exactly.

| rule | interpolant | weights |
|---|---|---|
| left Riemann | degree 0, left endpoint | $w_i = \Delta t_i$, $w_N = 0$ |
| **trapezoid** | degree 1, both endpoints | $w_i = \tfrac12(\Delta t_{i-1} + \Delta t_i)$ |
| Simpson | degree 2, three points | $\tfrac{1}{3}\Delta t\,(1,4,2,\dots,4,1)$, uniform $\Delta t$ |
| Gauss-Legendre | chooses nodes | weights at roots of Legendre polynomial $P_n$ |

Simpson's row carries one spacing because it presupposes a uniform grid.

### 1.2 Degree of exactness

An interpolatory rule is exact for polynomials up to degree of its interpolant (Davis & Rabinowitz, 1984, §2.1). Trapezoid: degree 1, by an argument local to each subinterval, hence valid on any grid.

Equal weights are exact on constants, and on lines only when nodes are symmetric about midpoint. `test_trapezoid_on_constants_and_lines`.

### 1.3 Error

For $g \in C^2$, with $h_{\max} = \max_i \Delta t_i$ (Davis & Rabinowitz, 1984),

$$\big|I(g) - Q_w(g)\big| \;\le\; \tfrac{T}{12}\,h_{\max}^2\,\|g''\|_\infty ,$$

by summing local error $-\tfrac1{12}h_i^3 g''(\xi_i)$. Accuracy is limited by curvature and by widest gap. §2 takes up $\|g''\|_\infty = \infty$.

### 1.4 Availability

| rule | exact to degree | requires |
|---|---|---|
| left Riemann | 0 | nothing |
| **trapezoid** | **1** | **nothing** |
| Simpson | 3 | uniform spacing, odd number of points |
| Gauss-Legendre | $2N-1$ | control of node positions |

Simpson's weights place middle node at midpoint; on unequal spacing coefficients change per triple, so standard formula becomes a different rule rather than an inaccurate one. Gauss-Legendre requires choosing where to sample.

With fixed nodes, **trapezoid is adopted**. Simpson and Gauss-Legendre may appear later as reference values on grids we control.

### 1.5 Cost: bias against variance

Weights chosen to remove bias cost variance. Let $Y_i = g(t_i) + \varepsilon_i$ with $\operatorname{Var}(\varepsilon_i) = \sigma^2$ independent, and $\tilde w_i = w_i/\sum_j w_j$. Then

$$\operatorname{Var}\Big(\sum_i \tilde w_i Y_i\Big) = \sigma^2 \sum_i \tilde w_i^{\,2},
\qquad \sum_i \tilde w_i^{\,2} \ge \frac1{N+1} \quad\text{(Cauchy-Schwarz)},$$

with equality iff weights are equal.

> Equal weights minimise variance among normalised linear estimators with independent equal-variance noise. Trapezoid is a consistent second-order quadrature rule for $C^2$ integrands on a refining grid. An unconditional optimal weighting follows from neither statement.

**Effective sample size.** An unweighted average of $\nu$ observations has variance $\sigma^2/\nu$. Setting $\sigma^2/\nu = \sigma^2\sum_i \tilde w_i^{\,2}$ gives Kish's

$$n_{\mathrm{eff}} = \frac{1}{\sum_i \tilde w_i^{\,2}} \le N+1 ,$$

the unweighted sample size of equal noise. Falls as weight concentrates: $N+1$ for equal weights, $1$ for all weight on one node. Clustered grid gives $n_{\mathrm{eff}} \ll N+1$, so §3's bias correction is paid for in variance.

Zhang & Wang (2016) show preferable weighting reverses with sampling density; Godambe (1955) proved no best linear unbiased estimator exists even under simple random sampling.

This notebook is noise-free throughout: §3 evaluates deterministic functions, so bias governs there.

Noise also biases. For $p=2$, $\mathbb{E}[Q_w(Y^2)] = \int_0^T f^2 + \sigma^2 T + O(h_{\max}^2)$, using $\sum_j w_j = T$. Term $\sigma^2 T$ is independent of $w$; removing it requires smoothing before integrating (Ramsay & Silverman, 2005, ch. 3 to 5).

### 1.6 Conclusion

$$\|f\|_{L^p}^p \;\approx\; \sum_{i=0}^N w^{\mathrm{trap}}_i\,|f(t_i)|^p ,$$

implemented as `integral_norm(t, x, p)` and `integral_distance(t, x, y, p)`.

Among rules using only adjacent fixed nodes, trapezoid is exact for affine integrands and is consistent when the maximum spacing tends to zero. It is the adopted default. Equal weights minimise variance under the noise model of §1.5; splines use stronger smoothness assumptions; smoothing addresses noisy observations; Godambe (1955) rules out an unconditional best linear unbiased estimator.

§3 does not depend on choice of rule. Any consistent estimator of $\int f^2$ shows the same failure of equal weights.

Carried forward: adopt trapezoid; make no noise-robustness claim; report $n_{\mathrm{eff}}$ with every loss value.

## 2. Convergence rate (uniform grid)

§1 chose weights. §2 fixes sample budget, so §3 cannot be attributed to quadrature error.

Grid here is **uniform**, spacing $h = T/N$ over $N$ intervals. Then $w^{\mathrm{unif}}$ and $w^{\mathrm{trap}}$ differ only at endpoints, so statements below cover both rules and choice of weights is invisible. §3 removes that restriction and the two separate.

**Rates.** For $g \in C^2$, trapezoid is $O(h^2)$ and left Riemann $O(h)$ (Davis & Rabinowitz, 1984, §2.1 and §2.4). `test_convergence_rates_on_non_periodic_integrand` asserts both orders for the implementation.

**Euler-Maclaurin.** Truncating at $M$ terms,

$$\int_a^b g \;=\; h\Big[\tfrac12 g_0 + \cdots + \tfrac12 g_N\Big] \;-\; \sum_{k=1}^{M}\frac{B_{2k}}{(2k)!}\,h^{2k}\Big[g^{(2k-1)}(b) - g^{(2k-1)}(a)\Big] \;+\; R_M ,
\qquad |R_M| \;\le\; \frac{2\zeta(2M)}{(2\pi)^{2M}}\,h^{2M}\int_a^b \big|g^{(2M)}\big| ,$$

with $B_{2k}$ Bernoulli numbers. Every retained term is a difference of odd derivatives at endpoints, so for smooth periodic $g$ all vanish and only $R_M$ survives, for every $M$: convergence exceeds any power of $h$ (Trefethen & Weideman, 2014). A convergence test on a periodic integrand therefore measures nothing, which is why $e^t$ is used rather than $\sin^2$. `test_trapezoid_on_smooth_periodic_integrand`.

**Without $C^2$.** The second-order bound is unavailable. If $g$ is $\alpha$-Hölder, a direct interval bound gives $O(N^{-\alpha})$ on a uniform grid. Sharper stochastic rates require assumptions on process and integrand; no general Brownian rate is used here.

**Budget.** A few hundred samples for smooth $g$, $10^3$ to $10^4$ for a rough one. Either way quadrature error sits orders below effect §3 measures, so it is controllable by sampling more densely. §3 turns to an error that is not.

## 3. Non-uniform sampling

§2 bounded $|Q_w(g) - I(g)|$ at finite $N$, which vanishes as the grid refines. This section concerns $\lim_m \mathrm{MSE} - \tfrac1T\int_0^T f^2$, which does not.

**Setup.** For $m = 1, 2, \dots$ let $\mathcal{T}_m = \{0 = t_0^{(m)} < \cdots < t_{N_m}^{(m)} = T\}$, with mesh $h_m = \max_i \Delta t_i^{(m)}$ and empirical measure $\mu_m = \frac{1}{N_m+1}\sum_i \delta_{t_i^{(m)}}$. Rules of §0.1 on this grid are

$$Q^{\mathrm{trap}}_m(g) = \sum_i w^{\mathrm{trap}}_i\, g\big(t_i^{(m)}\big),
\qquad
Q^{\mathrm{unif}}_m(g) = \frac{T}{N_m+1}\sum_i g\big(t_i^{(m)}\big) = T\!\int_0^T g\,\mathrm{d}\mu_m ,$$

with $\mathrm{MSE} = \tfrac1T Q^{\mathrm{unif}}_m(f^2)$. Both are instances of $Q_w$; only weights differ.

§3.1 gives the two limits, §3.2 a density realising the hypothesis of the second, §3.3 the same result in Lebesgue form.

### 3.1 The two limits

**Proposition 1.** *Let $g \in C([0,T])$. If $h_m \to 0$ then $Q^{\mathrm{trap}}_m(g) \to \int_0^T g$.*

Consistency of the trapezoid rule for continuous integrands. Hypothesis constrains mesh, not placement of nodes within it.

**Proposition 2.** *Let $g \in C([0,T])$ and $\mu_m \Rightarrow \mu$ weakly, $\mathrm{d}\mu = \rho\,\mathrm{d}t$ for a probability density $\rho$ on $[0,T]$. Then $Q^{\mathrm{unif}}_m(g) \to T\!\int_0^T g\rho$.*

Immediate from $Q^{\mathrm{unif}}_m(g) = T\!\int g\,\mathrm{d}\mu_m$. Both constructions of §3.2 satisfy the hypothesis.

**Corollary.** *Let $f \in C([0,T])$ and let both hypotheses hold. Then*

$$\underbrace{\tfrac1T\,Q^{\mathrm{trap}}_m(f^2)}_{\text{integral norm}} \;\longrightarrow\; \tfrac1T\!\int_0^T f^2 ,
\qquad
\underbrace{\tfrac1T\,Q^{\mathrm{unif}}_m(f^2)}_{\mathrm{MSE}} \;\longrightarrow\; \int_0^T f^2\rho ,$$

*the limits equal for every $f \in C([0,T])$ iff $\rho = 1/T$ a.e.*

Converse: if $\int_0^T f^2(\rho - \tfrac1T) = 0$ for every $f \in C([0,T])$, take $f = \sqrt g$ to get $\int_0^T g(\rho - \tfrac1T) = 0$ for every non-negative $g \in C([0,T])$, hence for every $g$ by linearity, hence $\rho = 1/T$ a.e.

> MSE integrates $f^2$ against the **sampling density**; the integral norm integrates it against **normalised Lebesgue measure**.

Accuracy is not what separates them. Left Riemann is first order and the crudest rule in §1.1, but its weights are spacings, so Proposition 1 applies verbatim and it converges to $\tfrac1T\int f^2$ on a clustered grid too (`test_limits_under_non_uniform_sampling`). Divide is whether weights see the grid.

Asymmetry between hypotheses is the content. Proposition 1 requires only $h_m \to 0$, so $Q^{\mathrm{trap}}_m$ is consistent on every refining grid. Proposition 2 requires $\mu_m$ to converge, and limit inherits whatever it converged to, so $Q^{\mathrm{unif}}_m$ is consistent only when $\mu_m \Rightarrow \mathrm{Unif}[0,T]$. Non-uniform nodes do not break quadrature: $\Delta t_i \approx 1/(N_m\rho(t_i))$ is exactly the correction $w^{\mathrm{trap}}$ applies. They break $w^{\mathrm{unif}}$, by a fixed amount rather than a vanishing one.

**A change of variables removes the bias.** Let $F$ be CDF of $\rho$ and $u = F(t)$. Then $\int_0^T g\rho\,\mathrm{d}t = \int_0^1 (g\circ F^{-1})\,\mathrm{d}u$ and nodes are uniform in $u$, so MSE is consistent in $u$-time. Bias is a coordinate artefact, removable when $\rho$ is known; reweighting is what remains when it is not.

**Any such estimator is a seminorm.** $Q_w(|f|^p)^{1/p}$ vanishes on every $f$ supported between the nodes, so for any weights depending only on $\mathcal{T}_m$ it is a seminorm on $C([0,T])$ with kernel $\{f : f|_{\mathcal{T}_m} = 0\}$. Unlike §3.1 no choice of $w$ removes this, so a small loss value certifies nothing without sampling density beside it. Refinement fixes it for fixed $f$ but never uniformly over $f$; that statement belongs to the separation question rather than this notebook.

> **Parallel with sFML.** A quantity differing between two models is invisible to the objective because data does not separate them.

### 3.2 A density realising the hypothesis

Proposition 2 needs $\mu_m \Rightarrow \rho\,\mathrm{d}t$. Independent draws from $\rho$ give one, by the strong law. So does the **quantile grid**: with $F$ the CDF of $\rho$,

$$t_i = F^{-1}(u_i), \qquad u_i = \tfrac{i}{N_m} ,$$

whose spacings satisfy $\Delta t_i = 1/\big(N_m\,\rho(t_i)\big) + O(N_m^{-2})$, realising $\rho$ to first order and without Monte Carlo noise. For a power-law $\rho$, $F^{-1}$ is a single power.

Take $\rho(t) = \tfrac1\alpha t^{1/\alpha - 1}$ on $[0,1]$, so $t_i = u_i^{\alpha}$, and $f(t) = ct$. Both limits are then closed-form and their ratio is

$$\frac{\int_0^1 f^2\rho}{\int_0^1 f^2} \;=\; \frac{3}{1 + 2\alpha} ,$$

independent of $c$. For $0 < \alpha \le 1$ ratio lies in $[1,3)$ and tends to $3$ as $\alpha \to 0$. `test_limits_under_non_uniform_sampling` checks both limits at $2^{20}$ points.

### 3.3 The same statement in Lebesgue form

Partitioning range rather than domain gives layer cake identity: with $\mu_f(\lambda) = \big|\{t \in [0,T] : |f(t)| > \lambda\}\big|$,

$$\int_0^T |f|^p \;=\; \int_0^\infty p\,\lambda^{p-1}\,\mu_f(\lambda)\,\mathrm{d}\lambda ,$$

so $\|f\|_{L^p}$ depends on $f$ only through $\mu_f$, equivalently through the decreasing rearrangement $f^*$ (Lieb & Loss, 2001, §1.13 and ch. 3).

**It is the same estimator.** With $\hat\mu(\lambda) = \sum_i w_i \mathbb{1}[|f(t_i)| > \lambda]$,

$$\int_0^\infty p\lambda^{p-1}\hat\mu(\lambda)\,\mathrm{d}\lambda
\;=\; \sum_i w_i \int_0^{|f(t_i)|} p\lambda^{p-1}\,\mathrm{d}\lambda
\;=\; Q_w(|f|^p) ,$$

exactly. Weights survive: measuring how long $f$ spends above a level is itself an integral in $t$.

**It gives the shortest form of §3.1.** Both estimators are the same functional of a pushforward measure, differing in which measure is pushed forward:

$$\tfrac1T Q^{\mathrm{trap}}_m(f^2) \to \int |y|^2 \,\mathrm{d}\big(f_*\lambda_T\big)(y),
\qquad
\mathrm{MSE} \to \int |y|^2 \,\mathrm{d}\big(f_*\rho\big)(y) ,$$

with $\lambda_T$ normalised Lebesgue measure on $[0,T]$. Corollary is then $f_*\lambda_T = f_*\rho$ for every $f$ iff $\rho = \lambda_T$.

**And a warning.** A norm seeing $f$ only through $\mu_f$ is invariant under every measure-preserving rearrangement of time: permute the path arbitrarily and $\|f\|_{L^p}$ is unchanged. Acceptable in a norm, disqualifying in a path-to-path loss, where order is the structure being learned. Project therefore does not stop at $L^p$; $p$-variation (notebook 02) is introduced to measure order.

## 4. Choice of $p$

`integral_norm(t, x, p)` takes any $p \ge 1$, and $p = \infty$ for the sup norm. §§1 to 3 hold for every $p$, fixing no value, with $g = |f|^p$ the integrand throughout.

On a probability space, and $[0,1]$ with Lebesgue measure is one, Lyapunov's inequality gives $\|f\|_{L^p} \le \|f\|_{L^q}$ for $p \le q$, with $\|f\|_{L^p} \to \sup_t |f(t)|$. Larger $p$ concentrates loss on worst-behaved part of path. `test_lp_monotonicity_in_p`, `test_sup_norm`. Normalisation $T^{1/p}$ in `integral_distance` makes values comparable across $p$.

Discussion takes $p = 2$: value comparable to MSE, and only $p$ for which $L^p$ is a Hilbert space, needed once inner-product structure enters. $p = \infty$ serves as a diagnostic.

## 5. Summary

$$\|f\|_{L^p}^p \;\approx\; Q^{\mathrm{trap}}_m\big(|f|^p\big) = \sum_{i=0}^N w^{\mathrm{trap}}_i\,|f(t_i)|^p , \qquad w^{\mathrm{trap}}_i = \tfrac12(\Delta t_{i-1} + \Delta t_i) ,$$

as `integral_norm(t, x, p)` and `integral_distance(t, x, y, p)`. Trapezoid is exact for affine integrands and has error $O(h^2)$ for $g \in C^2$ (§§1 to 2). With weaker regularity, rate depends on that regularity.

One result not settled by a citation: for $f \in C([0,T])$, with $h_m$ and $\mu_m$ as in §3,

$$h_m \to 0 \;\Longrightarrow\; \tfrac1T Q^{\mathrm{trap}}_m(f^2) \to \tfrac1T\!\int_0^T f^2 ,
\qquad
\mu_m \Rightarrow \rho\,\mathrm{d}t \;\Longrightarrow\; \mathrm{MSE} \to \int_0^T f^2\rho ,$$

equal for every $f$ iff $\rho = 1/T$ a.e. First hypothesis constrains mesh, second constrains where nodes go. So MSE is **inconsistent** for $\tfrac1T\|f\|_{L^2}^2$ under non-uniform sampling and refining does not help. Bias is removable by change of variables $u = F(t)$ when $\rho$ is known (§3.1); reweighting is what remains when it is not. Separately, and for any weights, the estimator is only a seminorm: it vanishes on functions living between nodes (§3.1).

**Literature context.** Quadrature on non-uniform grids is textbook (Davis & Rabinowitz, 1984; Trefethen & Weideman, 2014). Elapsed-time weighting uses the same density-compensation principle as inverse-probability estimators such as Horvitz--Thompson (1952), but trapezoid weights formed from neighbouring gaps are not generally a Horvitz--Thompson estimator. Section 3 restates the familiar bias of an unweighted average under non-uniform coverage. Zhang & Wang (2016) and Godambe (1955) show why weighting remains a bias--variance choice without an unconditional optimum; §3 studies a regime where bias dominates.

## 6. Role in the experimental chain

This notebook predicts a controlled contrast: equal sample weights and elapsed-time weights should behave alike on a uniform mesh, but can prefer different fitted paths when observation density is uneven. Notebook 03 contains an early reconstruction pilot. Notebook 05 tests the contrast while fitting one fixed path with a Neural ODE, and notebook 06 tests it while learning a Brownian-driver to OU-response operator. Both completed studies evaluate every trained model on a common dense uniform grid, so the comparison is about whole-interval path error rather than performance only at training samples.

Notebook 02 studies $p$-variation as a separate roughness diagnostic. Notebook 07 then demonstrates a limitation of coordinate integral norms: when supplied rough paths differ only in their Lévy-area coordinates, their observed coordinate paths and hence MSE, $J_2$, $L^1$ and $L^\infty$ discrepancies are exactly unchanged, while lift-aware signature discrepancies can respond. Together these notebooks separate three issues: the measure used to average error over time, whether derivatives are supervised, and whether representation contains higher-order ordered information.

## References cited in this notebook

Keyed to `../papers/references.bib`.

**Quadrature and numerical integration**
- **Davis & Rabinowitz (1984)**, *Methods of Numerical Integration*: error terms, Gauss-Legendre. §1, §2, §5.
- **Trefethen & Weideman (2014)**, *The exponentially convergent trapezoidal rule*, SIAM Review 56(3): periodic degeneracy. §2, §5.

**Measure theory**
- **Lieb & Loss (2001)**, *Analysis*, 2nd ed.: layer cake representation §1.13, rearrangement ch. 3. §3.3.

**Weighting and estimation**
- **Horvitz & Thompson (1952)**, JASA 47(260): inverse-probability weighting. §5.
- **Godambe (1955)**, *JRSS B* 17(2): no best linear unbiased estimator exists. §1.5, §5.
- **Zhang & Wang (2016)**, *Ann. Statist.* 44(5): efficiency of equal weighting reverses with sampling density. §1.5, §5.

**Functional data analysis**
- **Ferraty & Vieu (2006)**: discretised $L^2$ semi-metric, weights $w_j = t_j - t_{j-1}$. §1.
- **Ramsay & Silverman (2005)**, ch. 3 to 5: basis expansion, roughness penalties. §1.5. Route not yet taken.

**Irregular sampling in machine learning** (cited in §0.3 for what they leave alone)
- **Che et al. (2018)**, GRU-D, *Sci. Rep.* 8.
- **Rubanova, Chen & Duvenaud (2019)**, Latent ODEs, NeurIPS.
- **Shukla & Marlin (2021)**, mTAN, ICLR, and survey arXiv:2012.00168.
- **Kidger et al. (2020)**, Neural CDEs, NeurIPS.